# 03 — Baseline Seq2Seq Training

Train and evaluate the baseline encoder-decoder model (no attention).

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from nmt.models.encoder import Encoder
from nmt.models.decoder import Decoder
from nmt.models.seq2seq import Seq2Seq
from nmt.training.trainer import Trainer
from nmt.training.callbacks import EarlyStopping
from nmt.training.checkpoints import CheckpointManager
from nmt.utils.config import load_config
from nmt.utils.seed import set_seed
from nmt.utils.logging import setup_logging

setup_logging('INFO')
set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Load Config

In [ ]:
cfg = load_config('../configs/amharic.yaml')
print('Model type:', cfg['model']['type'])

## 2. Build Model

In [ ]:
# Replace SRC_VOCAB_SIZE / TGT_VOCAB_SIZE with your actual values
SRC_VOCAB_SIZE = 8000
TGT_VOCAB_SIZE = 8000
PAD_IDX = 0

encoder = Encoder(
    vocab_size=SRC_VOCAB_SIZE,
    embed_dim=cfg['model']['embed_dim'],
    hidden_dim=cfg['model']['hidden_dim'],
    num_layers=cfg['model']['num_layers'],
    dropout=cfg['model']['dropout'],
    bidirectional=cfg['model']['bidirectional_encoder'],
    padding_idx=PAD_IDX,
)
decoder = Decoder(
    vocab_size=TGT_VOCAB_SIZE,
    embed_dim=cfg['model']['embed_dim'],
    hidden_dim=cfg['model']['hidden_dim'],
    num_layers=cfg['model']['num_layers'],
    dropout=cfg['model']['dropout'],
    padding_idx=PAD_IDX,
)
model = Seq2Seq(encoder, decoder, src_pad_idx=PAD_IDX, tgt_pad_idx=PAD_IDX, device=DEVICE)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {num_params:,}')

## 3. Configure Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=cfg['training']['learning_rate'])
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
early_stopping = EarlyStopping(monitor='val_loss', patience=cfg['training']['early_stopping_patience'])
checkpoint_mgr = CheckpointManager(
    checkpoint_dir='../models/seq2seq',
    monitor='val_loss',
    mode='min',
)

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
    clip_grad_norm=cfg['training']['clip_grad_norm'],
    teacher_forcing_ratio=cfg['training']['teacher_forcing_ratio'],
    checkpoint_manager=checkpoint_mgr,
    callbacks=[early_stopping],
)

## 4. Train

> Replace `train_loader` and `val_loader` with real DataLoaders.

In [ ]:
# history = trainer.train(train_loader, val_loader, epochs=cfg['training']['epochs'])

## 5. Plot Training Curves

In [ ]:
# if history:
#     fig, ax = plt.subplots(figsize=(10, 4))
#     ax.plot(history['train_loss'], label='Train Loss')
#     ax.plot(history['val_loss'], label='Val Loss')
#     ax.set_xlabel('Epoch')
#     ax.set_ylabel('Loss')
#     ax.set_title('Baseline Seq2Seq — Training Loss')
#     ax.legend()
#     plt.tight_layout()
#     plt.savefig('../reports/figures/baseline_seq2seq_loss.png', dpi=150)
#     plt.show()